# CAPPIMU Data Read


## DATA

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from collections import defaultdict

# Set up paths
data_dir = Path('../data/raw/CAPPIMU/data')

def read_subject_trial(subject_id, data_dir=data_dir, verbose=False):
    """
    Read all sensor data files for a given subject.
    
    Parameters:
    -----------
    subject_id : int
        Subject ID (e.g., 1, 2, 3, ...)
    data_dir : Path
        Base directory containing subject folders
    verbose : bool
        If True, print detailed loading messages
        
    Returns:
    --------
    dict : Dictionary with sensor names as keys and DataFrames as values
    """
    subject_path = data_dir / f'subject_{subject_id}'
    
    # Better error message with actual path
    if not subject_path.exists():
        if verbose:
            print(f"❌ Subject path not found: {subject_path.absolute()}")
        return None
    
    if verbose:
        print(f"✓ Found subject directory: {subject_path}")
    
    # Common sensor files
    sensor_files = [
        'chest', 'arm_l', 'arm_r', 'head', 'knee_l', 'knee_r',
        'pocket', 'wrist_l', 'wrist_r', 'insole'
    ]
    
    # Read all available sensor files
    subject_data = {}
    for sensor in sensor_files:
        csv_path = subject_path / f'{sensor}.csv'
        if csv_path.exists():
            try:
                df = pd.read_csv(csv_path)
                subject_data[sensor] = df
                if verbose:
                    print(f"✓ Loaded {sensor}.csv: {len(df)} rows, {len(df.columns)} columns")
            except Exception as e:
                if verbose:
                    print(f"⚠️ Error reading {sensor}.csv: {e}")
    
    # Check for insole_ankle
    insole_ankle_path = subject_path / 'insole_ankle.csv'
    if insole_ankle_path.exists():
        try:
            df = pd.read_csv(insole_ankle_path)
            subject_data['insole_ankle'] = df
            if verbose:
                print(f"✓ Loaded insole_ankle.csv: {len(df)} rows, {len(df.columns)} columns")
        except Exception as e:
            if verbose:
                print(f"⚠️ Error reading insole_ankle.csv: {e}")
    
    # Read joint data if available
    joint_dir = subject_path / 'joint'
    if joint_dir.exists():
        joint_files = list(joint_dir.glob('*.csv'))
        for joint_file in joint_files:
            try:
                df = pd.read_csv(joint_file)
                joint_name = joint_file.stem
                subject_data[f'joint_{joint_name}'] = df
                if verbose:
                    print(f"✓ Loaded joint/{joint_name}.csv: {len(df)} rows, {len(df.columns)} columns")
            except Exception as e:
                if verbose:
                    print(f"⚠️ Error reading joint/{joint_name}.csv: {e}")
    
    return subject_data

# Find all available subjects
print(f"Data directory: {data_dir.absolute()}\n")
print("Finding all available subjects...")

all_subject_dirs = sorted([d for d in data_dir.glob('subject_*') if d.is_dir()])
subject_ids = [int(d.name.split('_')[1]) for d in all_subject_dirs]
subject_ids.sort()

print(f"Found {len(subject_ids)} subjects: {subject_ids}\n")
print(f"{'='*80}")
print(f"PROCESSING ALL SUBJECTS")
print(f"{'='*80}\n")

# Store summary information for all subjects
all_subjects_summary = []
all_subjects_data = {}

# Process each subject
for subject_id in subject_ids:
    print(f"\n{'='*80}")
    print(f"Processing Subject {subject_id}...")
    print(f"{'='*80}")
    
    try:
        subject_data = read_subject_trial(subject_id, verbose=True)
        
        if subject_data is None or len(subject_data) == 0:
            print(f"⚠️ No data loaded for subject {subject_id}")
            all_subjects_summary.append({
                'subject_id': subject_id,
                'status': 'failed',
                'num_sensors': 0,
                'sensor_names': [],
                'has_insole': False,
                'has_insole_ankle': False,
                'num_joint_files': 0
            })
            continue
        
        # Store data
        all_subjects_data[subject_id] = subject_data
        
        # Collect summary
        summary = {
            'subject_id': subject_id,
            'status': 'success',
            'num_sensors': len(subject_data),
            'sensor_names': list(subject_data.keys()),
            'has_insole': 'insole' in subject_data,
            'has_insole_ankle': 'insole_ankle' in subject_data,
            'num_joint_files': len([k for k in subject_data.keys() if k.startswith('joint_')])
        }
        
        # Add insole info if available
        if 'insole' in subject_data:
            summary['insole_rows'] = len(subject_data['insole'])
            summary['insole_columns'] = len(subject_data['insole'].columns)
            summary['insole_missing'] = subject_data['insole'].isnull().sum().sum()
        
        all_subjects_summary.append(summary)
        
        print(f"\n✓ Subject {subject_id} processed successfully")
        print(f"  - Total sensors: {summary['num_sensors']}")
        print(f"  - Has insole: {summary['has_insole']}")
        print(f"  - Has insole_ankle: {summary['has_insole_ankle']}")
        print(f"  - Joint files: {summary['num_joint_files']}")
        
    except Exception as e:
        print(f"\n❌ Error processing subject {subject_id}: {e}")
        all_subjects_summary.append({
            'subject_id': subject_id,
            'status': 'error',
            'error': str(e)
        })

# Create summary DataFrame
print(f"\n\n{'='*80}")
print(f"SUMMARY FOR ALL SUBJECTS")
print(f"{'='*80}\n")

summary_df = pd.DataFrame(all_subjects_summary)
print(summary_df.to_string(index=False))

# Statistics
print(f"\n\n{'='*80}")
print(f"OVERALL STATISTICS")
print(f"{'='*80}\n")

successful = summary_df[summary_df['status'] == 'success']
print(f"Total subjects: {len(summary_df)}")
print(f"Successfully processed: {len(successful)}")
print(f"Failed: {len(summary_df) - len(successful)}")

if len(successful) > 0:
    print(f"\nSensor statistics:")
    print(f"  - Average sensors per subject: {successful['num_sensors'].mean():.1f}")
    print(f"  - Min sensors: {successful['num_sensors'].min()}")
    print(f"  - Max sensors: {successful['num_sensors'].max()}")
    print(f"  - Subjects with insole: {successful['has_insole'].sum()}")
    print(f"  - Subjects with insole_ankle: {successful['has_insole_ankle'].sum()}")
    print(f"  - Average joint files: {successful['num_joint_files'].mean():.1f}")

# Display detailed info for first subject with insole data
print(f"\n\n{'='*80}")
print(f"DETAILED INSOLE DATA PREVIEW (First Subject with Insole)")
print(f"{'='*80}\n")

first_subject_with_insole = None
for subject_id in subject_ids:
    if subject_id in all_subjects_data and 'insole' in all_subjects_data[subject_id]:
        first_subject_with_insole = subject_id
        break

if first_subject_with_insole:
    print(f"Subject {first_subject_with_insole} Insole Data:")
    insole_df = all_subjects_data[first_subject_with_insole]['insole']
    print(f"\nShape: {insole_df.shape}")
    print(f"Columns: {list(insole_df.columns)}")
    print(f"\nFirst 5 rows:")
    print(insole_df.head())
    print(f"\nData types:\n{insole_df.dtypes}")
    print(f"\nMissing values:\n{insole_df.isnull().sum()}")
else:
    print("No subject with insole data found!")

# After loading insole data, add:
if insole_df.shape[0] == 0:
    print(f"⚠️ Warning: Insole data is empty!")


# At the end, add:
summary_df.to_csv('../data/processed/dataset_summary.csv', index=False)
print("\n✓ Summary saved to dataset_summary.csv")

ModuleNotFoundError: No module named 'pandas'